# Road Scene Understanding & Multi-Class Object Recognition using YOLOv8
**College PBL project: runs on a CPU laptop (no GPU needed)**

This notebook demonstrates YOLOv8 road-object detection, scene understanding, video analysis, evaluation, and optional KITTI fine-tuning.

In [ ]:
# Run once if needed (then restart the kernel):
# %pip install ultralytics opencv-python matplotlib

## 1. Setup and load YOLOv8

In [ ]:
import os, time, urllib.request
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

MODEL_NAME = 'yolov8n.pt'
CONF = 0.35
IMGSZ = 640
model = YOLO(MODEL_NAME)
ROAD_CLASSES = {'person','bicycle','car','motorcycle','bus','truck','traffic light','stop sign','van','pedestrian','cyclist','tram'}
VULNERABLE = {'person','bicycle','motorcycle','pedestrian','cyclist'}
VEHICLES = {'car','bus','truck','van','tram'}

def road_ids(m):
    return [i for i, n in m.names.items() if n in ROAD_CLASSES]

print('Classes:', len(model.names))
print('Road classes:', {i: model.names[i] for i in road_ids(model)})

## 2. Detect objects in a road image
Put a road image in the notebook folder as `street.jpg`; otherwise a sample image is downloaded.

In [ ]:
IMAGE_PATH = 'street.jpg'
SAMPLE_URL = 'https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg'
if not os.path.exists(IMAGE_PATH):
    urllib.request.urlretrieve(SAMPLE_URL, IMAGE_PATH)

def detect(src, conf=CONF, m=None):
    m = m or model
    r = m(src, conf=conf, imgsz=IMGSZ, classes=road_ids(m), verbose=False)[0]
    dets = [{'name': r.names[int(b.cls)], 'conf': float(b.conf), 'box': b.xyxy[0].tolist()} for b in r.boxes]
    return r, dets

t0 = time.time()
result, dets = detect(IMAGE_PATH)
print(f'Inference took {time.time()-t0:.2f}s - found {len(dets)} road objects')
plt.figure(figsize=(8,8))
plt.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
plt.axis('off'); plt.title('YOLOv8 detections'); plt.show()
for d in dets:
    print(f"{d['name']:14s} conf={d['conf']:.2f} box={[round(v) for v in d['box']]}")

## 3. Scene understanding
Objects are classified as left/ahead/right and near/far using image geometry. Simple hazard rules generate BRAKE, CAUTION, KEEP DISTANCE, and information alerts.

In [ ]:
def analyze_scene(dets, shape):
    H, W = shape[:2]
    rows, warnings = [], []
    for d in dets:
        x1, y1, x2, y2 = d['box']
        cx = (x1 + x2) / 2 / W
        zone = 'left' if cx < 1/3 else ('ahead' if cx < 2/3 else 'right')
        near = (y2 / H > 0.75) or ((y2-y1) / H > 0.35)
        rows.append((d['name'], zone, 'near' if near else 'far', d['conf']))
        if d['name'] in VULNERABLE and zone == 'ahead' and near:
            warnings.append(f"BRAKE: {d['name']} close ahead")
        elif d['name'] in VULNERABLE and zone == 'ahead':
            warnings.append(f"CAUTION: {d['name']} ahead")
        elif d['name'] in VEHICLES and zone == 'ahead' and near:
            warnings.append(f"KEEP DISTANCE: {d['name']} close ahead")
        elif d['name'] == 'stop sign':
            warnings.append('INFO: stop sign detected')
        elif d['name'] == 'traffic light':
            warnings.append('INFO: traffic light detected')
    counts = {}
    for n, *_ in rows:
        counts[n] = counts.get(n, 0) + 1
    summary = ('Scene: ' + ', '.join(f'{v} {k}' for k,v in counts.items())) if counts else 'No road objects detected.'
    return rows, sorted(set(warnings)), summary

img = cv2.imread(IMAGE_PATH)
rows, warnings, summary = analyze_scene(dets, img.shape)
print(summary)
for row in rows: print(row)
print('Alerts:', warnings or 'none')

In [ ]:
# Visualise scene zones
H, W = img.shape[:2]
plt.figure(figsize=(8,8))
plt.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
for x in (W/3, 2*W/3): plt.axvline(x, color='cyan', linestyle='--')
plt.axhline(0.75*H, color='orange', linestyle=':')
plt.text(10,30,'LEFT',color='cyan'); plt.text(W/3+10,30,'AHEAD',color='cyan'); plt.text(2*W/3+10,30,'RIGHT',color='cyan')
plt.axis('off'); plt.title("Zones and 'near' line"); plt.show()

## 4. Video inference
Process every few frames, save an annotated video, and plot object counts over time.

In [ ]:
VIDEO_PATH = 'road.mp4'
STRIDE = 3
MAX_FRAMES = 300

def process_video(path, out_path='annotated.mp4', conf=CONF, stride=STRIDE, max_frames=MAX_FRAMES):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps/max(stride,1), (W,H))
    counts=[]; i=0
    while i < max_frames:
        ok, frame = cap.read()
        if not ok: break
        if i % stride == 0:
            r=model(frame,conf=conf,imgsz=IMGSZ,classes=road_ids(model),verbose=False)[0]
            out.write(r.plot())
            c={}
            for b in r.boxes:
                n=r.names[int(b.cls)]; c[n]=c.get(n,0)+1
            counts.append(c)
        i += 1
    cap.release(); out.release()
    return counts

if os.path.exists(VIDEO_PATH):
    counts=process_video(VIDEO_PATH)
    print(f'Processed {len(counts)} frames -> annotated.mp4')
    names=sorted({n for c in counts for n in c})
    plt.figure(figsize=(10,3.5))
    for n in names: plt.plot([c.get(n,0) for c in counts], label=n)
    plt.xlabel('processed frame'); plt.ylabel('objects'); plt.title('Objects detected over time'); plt.legend(); plt.show()
else:
    counts=[]
    print(f"Add a video called '{VIDEO_PATH}' next to this notebook and rerun.")

In [ ]:
# Count unique objects with YOLO tracking
def count_unique(path, conf=CONF, max_frames=MAX_FRAMES):
    seen={}
    stream=model.track(source=path,conf=conf,imgsz=IMGSZ,classes=road_ids(model),persist=True,stream=True,verbose=False)
    for k,r in enumerate(stream):
        if r.boxes.id is not None:
            for tid,c in zip(r.boxes.id.int().tolist(),r.boxes.cls.int().tolist()):
                seen.setdefault(r.names[c],set()).add(tid)
        if k >= max_frames: break
    return {n:len(s) for n,s in seen.items()}

if os.path.exists(VIDEO_PATH):
    try: print('Unique objects:', count_unique(VIDEO_PATH))
    except Exception as e: print('Tracking skipped:', e)

## 5. Evaluation
Precision, recall, mAP50, and mAP50-95 are reported using the small COCO8 validation set as a pipeline demonstration.

In [ ]:
val=model.val(data='coco8.yaml',imgsz=IMGSZ,device='cpu',plots=False,verbose=False)
print(f'Precision {val.box.mp:.3f} | Recall {val.box.mr:.3f} | mAP50 {val.box.map50:.3f} | mAP50-95 {val.box.map:.3f}')
print('Per-class mAP50-95:')
for c,ap in zip(val.box.ap_class_index,val.box.maps[val.box.ap_class_index]):
    mark=' <- road class' if val.names[int(c)] in ROAD_CLASSES else ''
    print(f'{val.names[int(c)]:14s} {ap:.3f}{mark}')

## 6. Optional fine-tuning on KITTI
KITTI adapts the detector to driving scenes. For CPU use a small fraction and few epochs; a Colab GPU is recommended for serious training.

In [ ]:
RUN_FINETUNE=False
FT_DATA='kitti.yaml'
FT_EPOCHS=3
FT_FRACTION=0.1

if RUN_FINETUNE:
    ft_model=YOLO(MODEL_NAME)
    ft_model.train(data=FT_DATA,epochs=FT_EPOCHS,imgsz=416,batch=8,fraction=FT_FRACTION,device='cpu',workers=2,plots=True,project='runs',name='road_ft',exist_ok=True)
    best_path='runs/road_ft/weights/best.pt'
    best=YOLO(best_path)
    m_ft=best.val(data=FT_DATA,imgsz=416,device='cpu',plots=False,verbose=False)
    print(f'FINE-TUNED -> Precision {m_ft.box.mp:.3f} | Recall {m_ft.box.mr:.3f} | mAP50 {m_ft.box.map50:.3f} | mAP50-95 {m_ft.box.map:.3f}')
    r_ft,dets_ft=detect(IMAGE_PATH,m=best)
    print(analyze_scene(dets_ft,img.shape)[2])
else:
    print('Fine-tuning skipped. Set RUN_FINETUNE=True to run it.')

## 7. Results and discussion
Fill in the metrics after running the notebook.

| Item | Result |
|---|---|
| Model | YOLOv8n |
| CPU inference time per image | ___ s |
| Precision / Recall | ___ / ___ |
| mAP50 / mAP50-95 | ___ / ___ |
| Fine-tuned mAP50-95 on KITTI | ___ |

Limitations: distance and zone rules use image geometry rather than real depth. Night, rain, small objects, and occlusion can reduce accuracy. Future work can add lane detection, monocular depth, traffic-sign training, ONNX export, and webcam inference.

## References
- Redmon et al., *You Only Look Once: Unified, Real-Time Object Detection*, CVPR 2016.
- Ultralytics YOLO documentation: https://docs.ultralytics.com
- Geiger et al., *The KITTI Vision Benchmark Suite*, CVPR 2012.
- Lin et al., *Microsoft COCO*, ECCV 2014.